In [0]:
%run ../config/config_init

In [0]:
domain_name = create_widget("domain_name", "")
job_id = create_widget("job_id")
job_run_id = create_widget("job_run_id")
task_run_id = create_widget("task_run_id")
orchestrator_job_id = create_widget("orchestrator_job_id")
orchestrator_job_run_id = create_widget("orchestrator_job_run_id")

domain_name = dbutils.widgets.get("domain_name")
job_id = dbutils.widgets.get("job_id")
job_run_id = dbutils.widgets.get("job_run_id")
task_run_id = dbutils.widgets.get("task_run_id")
orchestrator_job_id = dbutils.widgets.get("orchestrator_job_id")
orchestrator_job_run_id = dbutils.widgets.get("orchestrator_job_run_id")

In [0]:
rows = (
    spark.table(f"{catalog_name}.{schema_config}.configurations_table")
         .filter(f"domain_name = '{domain_name}' AND medallion_layer = 'silver'")
         .collect()
)

for row in rows:
    # Get schema from the target table
    target_schema = spark.table(row.target).schema

    try:
        source_df = spark.table(row.source)

        # Drop duplicates + rows with all NULLs
        try:
            print("Deduping data...")
            source_df = spark.table(row.source)

            if hasattr(row, "source_keys") and row.source_keys:
                dedup_keys = [k.strip() for k in row.source_keys.split(",")]
                source_df = source_df.dropDuplicates(dedup_keys)
            else:
                # Fallback: dedupe on all columns if no keys provided
                source_df = source_df.dropDuplicates()
        except Exception as e:
            raise Exception(f"Error deduping {row.source}: {e}.")

        # Drop rows with all NULLs
        print("Dropping null rows...")
        source_df = source_df.dropna(how="all")

        # Trim all string columns
        for field in target_schema.fields:
            if isinstance(field.dataType, T.StringType):
                source_df = source_df.withColumn(field.name, F.trim(F.col(field.name)))

        # Cast columns to match target schema
        print("Casting columns to target schema...")
        for field in target_schema.fields:
            if field.name in source_df.columns:
                source_df = source_df.withColumn(
                    field.name,
                    F.col(field.name).cast(field.dataType)
                )

        # Add metadata columns
        source_df = (
            source_df.withColumn("create_date", F.current_timestamp())
                     .withColumn("source_name", F.lit(row.source))
                     .withColumn("meta_job_id", F.lit(job_id))
                     .withColumn("meta_job_run_id", F.lit(job_run_id))
                     .withColumn("meta_task_run_id", F.lit(task_run_id))
                     .withColumn("meta_orchestrator_job_id", F.lit(orchestrator_job_id))
                     .withColumn("meta_orchestrator_job_run_id", F.lit(orchestrator_job_run_id))
        )

        # Reorder columns to match target schema
        target_columns = [f.name for f in target_schema.fields]
        final_df = source_df.select(*target_columns)

        if row.operation == "append":
            final_df = final_df.filter(
                (F.col("meta_orchestration_job_id") == job_id) &
                (F.col("meta_orchestration_job_run_id") == job_run_id)
            )

    except Exception as e:
        raise Exception(f"Error processing {row.source}: {e}.")

    try:
        # Write to target table
        print(f"Writing to {row.target}...")
        write_table(final_df, row.target, row.operation, row.source_keys)
        print(f"Done writing to {row.target}.")
    except Exception as e:
        raise Exception(f"Error writing to {row.target}: {e}.")